In [1]:
import pandas as pd 
import numpy as np

In [2]:
data = pd.read_csv("cleaned_wind_data.csv")

In [3]:
data.head(5)

,STATION,NAME,LATITUDE,LONGITUDE,ELEVATION,DATE,wind_direction_deg,wind_speed_mps,tmp_deg,power_output_kw
0,43225099999,"KARWAR, IN",14.783333,74.133333,4.0,2024-04-20 03:00:00,180.0,4.1,29.0,50
1,43225099999,"KARWAR, IN",14.783333,74.133333,4.0,2024-05-16 12:00:00,340.0,4.1,31.8,50
2,43225099999,"KARWAR, IN",14.783333,74.133333,4.0,2024-05-24 12:00:00,320.0,5.1,33.4,100
3,43225099999,"KARWAR, IN",14.783333,74.133333,4.0,2024-05-30 09:00:00,290.0,4.1,36.6,50
4,43225099999,"KARWAR, IN",14.783333,74.133333,4.0,2024-06-08 12:00:00,320.0,5.1,26.2,100


In [4]:
# extracting Time features
data["DATE"] = pd.to_datetime(data["DATE"])

data["hour"] = data["DATE"].dt.hour
data["month"] = data["DATE"].dt.month
data["day"] = data["DATE"].dt.day
data["week_day"] = data["DATE"].dt.weekday
data["season"] = data["month"] % 12 // 3 + 1
data["temp_wind_interaction"] = data["tmp_deg"] * data["wind_direction_deg"]
data["hour_season_interaction"] = data["hour"] * data["season"]

data.head()

,STATION,NAME,LATITUDE,LONGITUDE,ELEVATION,DATE,wind_direction_deg,wind_speed_mps,tmp_deg,power_output_kw,hour,month,day,week_day,season,temp_wind_interaction,hour_season_interaction
0,43225099999,"KARWAR, IN",14.783333,74.133333,4.0,2024-04-20 03:00:00,180.0,4.1,29.0,50,3,4,20,5,2,5220.0,6
1,43225099999,"KARWAR, IN",14.783333,74.133333,4.0,2024-05-16 12:00:00,340.0,4.1,31.8,50,12,5,16,3,2,10812.0,24
2,43225099999,"KARWAR, IN",14.783333,74.133333,4.0,2024-05-24 12:00:00,320.0,5.1,33.4,100,12,5,24,4,2,10688.0,24
3,43225099999,"KARWAR, IN",14.783333,74.133333,4.0,2024-05-30 09:00:00,290.0,4.1,36.6,50,9,5,30,3,2,10614.0,18
4,43225099999,"KARWAR, IN",14.783333,74.133333,4.0,2024-06-08 12:00:00,320.0,5.1,26.2,100,12,6,8,5,3,8384.0,36


In [7]:
from sklearn.model_selection import train_test_split

# feature selection
train_parts = []
test_parts = []

for station_id, group in data.groupby("STATION"):
    if len(group) < 2:
        # If only 1 row, put it into the training set (or you can skip it)
        train_parts.append(group)
        continue
    x_station = group.drop("wind_speed_mps", axis=1)
    y_station = group["wind_speed_mps"]

    x_train_s, x_test_s, y_train_s, y_test_s = train_test_split(
        x_station, y_station, test_size=0.2, shuffle=True, random_state=42
    )

    train_parts.append(pd.concat([x_train_s, y_train_s], axis=1))
    test_parts.append(pd.concat([x_test_s, y_test_s], axis = 1))

train_df = pd.concat(train_parts).reset_index(drop=True)
test_df = pd.concat(test_parts).reset_index(drop=True)

In [21]:
# train test split 

x_train, y_train = train_df.drop(["wind_speed_mps", "NAME", "DATE", "power_output_kw"], axis=1), train_df["wind_speed_mps"]
x_test, y_test = test_df.drop(["wind_speed_mps", "NAME", "DATE", "power_output_kw"], axis=1), test_df["wind_speed_mps"]

In [22]:
from sklearn.ensemble import RandomForestRegressor

# model selection
model = RandomForestRegressor(n_estimators = 100, random_state = 42)

model.fit(x_train, y_train)

,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [23]:
# model evaluation 

from sklearn.metrics import mean_absolute_error, r2_score , mean_squared_error

y_predict = model.predict(x_test)

In [24]:

print(f"actual value \n{y_test}\n")
print(f"predicted value \n{y_predict}")

actual value 
0       7.2
1       4.6
2       5.1
3       3.6
4       4.1
       ... 
3536    4.6
3537    3.6
3538    4.1
3539    4.1
3540    3.6
Name: wind_speed_mps, Length: 3541, dtype: float64

predicted value 
[5.86316667 5.1        5.08825    ... 4.498      4.22516667 3.765     ]


In [25]:
print(f"MAE: {mean_absolute_error(y_test, y_predict)}")
print(f"r2 score: {r2_score(y_test, y_predict)}")
print(f"RMSE: {mean_squared_error(y_test, y_predict)}")

MAE: 0.6776262769910757
r2 score: 0.45095348630658527
RMSE: 0.8380012028635129


In [26]:
# Returns power output in kW for a given wind speed in m/s based on the discrete power curve of the Suzlon S66 turbine.
def suzlon_s66_power_output(wind_speed):
    if wind_speed < 4:
        return 0
    elif wind_speed < 5:
        return 50
    elif wind_speed < 6:
        return 100
    elif wind_speed < 7:
        return 200
    elif wind_speed < 8:
        return 350
    elif wind_speed < 9:
        return 550
    elif wind_speed < 10:
        return 800
    elif wind_speed < 11:
        return 1000
    elif wind_speed < 12:
        return 1150
    elif wind_speed < 13:
        return 1225
    elif wind_speed < 25:
        return 1250
    else:
        return 0

In [28]:
power_predict = [suzlon_s66_power_output(speed) for speed in y_predict]

In [29]:
import joblib
joblib.dump(model, "wind_speed_model.pkl")

['wind_speed_model.pkl']